# CSCI E-89 Homework 03, Problem 5

**Jawad Hussein**


## Script 1 — Load the data

Download Fashion-MNIST, scale images to float tensors in `[0, 1]`, and reproducibly split the original training data into 55,000 training and 5,000 validation images.


In [ ]:
import torch
from torch.utils.data import random_split
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor

RANDOM_SEED = 42
DATA_DIR = "data"

# ToTensor converts each image to float32 and scales pixel values to [0, 1].
full_training_set = FashionMNIST(DATA_DIR, train=True, download=True, transform=ToTensor())
test_set = FashionMNIST(DATA_DIR, train=False, download=True, transform=ToTensor())

split_generator = torch.Generator().manual_seed(RANDOM_SEED)
training_set, validation_set = random_split(
    full_training_set, [55_000, 5_000], generator=split_generator
)

print(f"Training images: {len(training_set):,}")
print(f"Validation images: {len(validation_set):,}")
print(f"Test images: {len(test_set):,}")


## Script 2 — Create DataLoaders

Create loaders with batches of 32, shuffle only the training split, and inspect the first training sample.


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
# A dedicated generator makes the shuffled training order reproducible.
loader_generator = torch.Generator().manual_seed(RANDOM_SEED)
training_loader = DataLoader(
    training_set, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator
)
validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

first_image, first_label = training_set[0]
class_names = full_training_set.classes
print(f"First training sample shape: {first_image.shape}")
print(f"First training sample data type: {first_image.dtype}")
print(f"First training sample class name: {class_names[first_label]}")


## Script 3 — Build the classifier

Define the 784–300–100–10 fully connected classifier with ReLU activations, choose GPU when available, and create the cross-entropy loss.


In [ ]:
from torch import nn

class FashionMNISTClassifier(nn.Module):
    """A fully connected classifier for 28-by-28 grayscale images."""

    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 300),
            nn.ReLU(),
            nn.Linear(300, 100),
            nn.ReLU(),
            nn.Linear(100, 10),
        )

    def forward(self, images):
        return self.network(images)

# Seed before model construction for reproducible initial weights.
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FashionMNISTClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
print(model)
print(f"Using device: {device}")


## Script 4 — Train the model

Train for 20 epochs with SGD at learning rate 0.1, use TorchMetrics to track overall accuracy, print each epoch’s metrics, and retain them in `history`.


In [ ]:
from torchmetrics.classification import MulticlassAccuracy

def train_model(model_to_train, train_loader, valid_loader, *, epochs=20, learning_rate=0.1):
    """Train a classifier and return epoch loss and accuracy histories."""
    model_to_train.to(device)
    optimizer = torch.optim.SGD(model_to_train.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    train_accuracy = MulticlassAccuracy(num_classes=10, average="micro").to(device)
    valid_accuracy = MulticlassAccuracy(num_classes=10, average="micro").to(device)
    results = {"loss": [], "training_accuracy": [], "validation_accuracy": []}

    for epoch in range(1, epochs + 1):
        model_to_train.train()
        total_loss = 0.0
        total_examples = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model_to_train(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_examples += labels.size(0)
            train_accuracy.update(logits.detach(), labels)

        epoch_loss = total_loss / total_examples
        epoch_train_accuracy = train_accuracy.compute().item()
        train_accuracy.reset()

        model_to_train.eval()
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                valid_accuracy.update(model_to_train(images), labels)
        epoch_valid_accuracy = valid_accuracy.compute().item()
        valid_accuracy.reset()

        results["loss"].append(epoch_loss)
        results["training_accuracy"].append(epoch_train_accuracy)
        results["validation_accuracy"].append(epoch_valid_accuracy)
        print(
            f"Epoch {epoch:02d}/{epochs} | Loss: {epoch_loss:.4f} | "
            f"Training accuracy: {epoch_train_accuracy:.4f} | "
            f"Validation accuracy: {epoch_valid_accuracy:.4f}"
        )
    return results

history = train_model(model, training_loader, validation_loader)


## Script 5 — Make predictions

For the first three validation images, compare predicted and actual names, print every class probability and the top four, and report the model’s parameter count.


In [ ]:
def count_parameters(model_to_count):
    """Count all trainable and non-trainable model parameters."""
    return sum(parameter.numel() for parameter in model_to_count.parameters())

images, labels = next(iter(validation_loader))
images = images[:3].to(device)
labels = labels[:3]
model.eval()
with torch.no_grad():
    probabilities = torch.softmax(model(images), dim=1).cpu()

print(f"Total model parameters: {count_parameters(model):,}")
for image_number, (label, class_probabilities) in enumerate(zip(labels, probabilities), 1):
    predicted_index = class_probabilities.argmax().item()
    print(f"\nValidation image {image_number}")
    print(f"Predicted: {class_names[predicted_index]}")
    print(f"Actual:    {class_names[label.item()]}")
    print("Probability for each class:")
    for class_name, probability in zip(class_names, class_probabilities):
        print(f"  {class_name:<12} {probability.item():.2%}")

    top_probabilities, top_indices = torch.topk(class_probabilities, 4)
    print("Top 4 most likely classes:")
    for rank, (class_index, probability) in enumerate(zip(top_indices, top_probabilities), 1):
        print(f"  {rank}. {class_names[class_index.item()]:<12} {probability.item():.2%}")


## Script 6 — Plot training accuracy

Plot training and validation accuracy from the history returned by the training function.


In [ ]:
import matplotlib.pyplot as plt

# Plot both curves so training progress and generalization are easy to compare.
epoch_numbers = range(1, len(history["training_accuracy"]) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epoch_numbers, history["training_accuracy"], label="Training accuracy")
plt.plot(epoch_numbers, history["validation_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fashion-MNIST Training Accuracy")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Script 7 — Optuna hyperparameter search

Run five seeded trials, tuning a log-scaled learning rate and a shared hidden-layer width. Train each configuration for 10 epochs and score its best validation accuracy.


In [ ]:
import optuna

class TunableFashionMNISTClassifier(nn.Module):
    """Classifier with the same tunable width for both hidden layers."""

    def __init__(self, hidden_neurons):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(28 * 28, hidden_neurons), nn.ReLU(),
            nn.Linear(hidden_neurons, hidden_neurons), nn.ReLU(),
            nn.Linear(hidden_neurons, 10),
        )

    def forward(self, images):
        return self.network(images)

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    hidden_neurons = trial.suggest_int("hidden_neurons", 20, 300)
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    trial_model = TunableFashionMNISTClassifier(hidden_neurons)
    trial_history = train_model(
        trial_model, training_loader, validation_loader,
        epochs=10, learning_rate=learning_rate,
    )
    return max(trial_history["validation_accuracy"])

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)
print(f"Best parameters: {study.best_params}")
print(f"Best validation accuracy: {study.best_value:.4f}")


## Script 8 — Optuna search with pruning

Pass the loaders explicitly into the objective, train one epoch at a time, report validation accuracy after each epoch, and let a median pruner stop weak trials during a 20-trial search.


In [ ]:
def pruning_objective(trial, train_loader, valid_loader):
    """Train one sampled configuration and expose epoch results for pruning."""
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    hidden_neurons = trial.suggest_int("hidden_neurons", 20, 300)
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    trial_model = TunableFashionMNISTClassifier(hidden_neurons).to(device)
    optimizer = torch.optim.SGD(trial_model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    best_validation_accuracy = 0.0

    for epoch in range(10):
        trial_model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(trial_model(images), labels)
            loss.backward()
            optimizer.step()

        trial_model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                predictions = trial_model(images).argmax(dim=1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)
        validation_accuracy = correct / total
        best_validation_accuracy = max(best_validation_accuracy, validation_accuracy)
        print(
            f"Trial {trial.number:02d} | Epoch {epoch + 1:02d}/10 | "
            f"Validation accuracy: {validation_accuracy:.4f}"
        )
        trial.report(validation_accuracy, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_validation_accuracy

pruning_sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
median_pruner = optuna.pruners.MedianPruner()
pruning_study = optuna.create_study(
    direction="maximize", sampler=pruning_sampler, pruner=median_pruner
)
pruning_study.optimize(
    lambda trial: pruning_objective(trial, training_loader, validation_loader),
    n_trials=20,
)
print(f"Best parameters: {pruning_study.best_params}")
print(f"Best validation accuracy: {pruning_study.best_value:.4f}")


# Fashion-MNIST Dialog Summary

Over the course of our dialog, we built a sequence of eight Python scripts that
form a complete Fashion-MNIST workflow. The scripts progress from preparing the
data, through defining and training a neural network, to inspecting results and
tuning hyperparameters. Each script can be run directly, while its functions and
classes can also be imported by the later scripts.

## 1. `prob5/script1_load_dataset.py` — Load and split the dataset

The first script downloads the Fashion-MNIST training and test datasets and
converts each image to a PyTorch tensor. It reproducibly splits the original
60,000-image training set into 55,000 training examples and 5,000 validation
examples using a generator seeded with `42`. Its `load_datasets` function returns
the training, validation, and test datasets, and its command-line output reports
the size of each split.

## 2. `prob5/script2_dataloaders.py` — Build data loaders

The second script calls `load_datasets` and wraps all three dataset splits in
PyTorch `DataLoader` objects with a batch size of 32. It shuffles only the
training data, leaving validation and test data in a stable order. When run
directly, it inspects the first training example and prints its tensor shape,
data type, and human-readable Fashion-MNIST class name.

## 3. `prob5/script3_model.py` — Define the classifier

The third script defines `FashionMNISTClassifier`, a fully connected neural
network for 28-by-28 grayscale images. The network flattens each image's 784
pixels, passes them through hidden layers of 300 and 100 neurons with ReLU
activations, and produces logits for the 10 clothing classes. The script seeds
PyTorch for reproducibility, selects CUDA when available (otherwise CPU), creates
a shared model instance, and defines cross-entropy as the classification loss.

## 4. `prob5/script4_train.py` — Train and validate the model

The fourth script adds the reusable `train_model` function. By default, it trains
for 20 epochs with stochastic gradient descent and a learning rate of 0.1. For
each epoch, it performs the full forward/backward optimization loop, measures
multiclass training accuracy, evaluates validation accuracy without calculating
gradients, and prints the epoch's metrics. It returns a history dictionary
containing loss, training accuracy, and validation accuracy so later scripts can
reuse the results.

## 5. `prob5/script5_predictions.py` — Inspect predictions

The fifth script trains the classifier and then examines the first three images
from the validation loader. It reports the model's total parameter count and, for
each selected image, prints the predicted class, actual class, probability for
every Fashion-MNIST class, and the four most likely classes in ranked order. The
probabilities are obtained by applying softmax to the model's logits while the
model is in evaluation mode.

## 6. `prob5/script6_plot_accuracy.py` — Visualize training progress

The sixth script trains the model and plots the recorded training and validation
accuracy for every epoch with Matplotlib. The chart includes labeled axes, a
title, a legend, a grid, and an accuracy range fixed from zero to one, making it
easy to compare learning performance and generalization over time.

## 7. `prob5/script7_optuna.py` — Tune hyperparameters

The seventh script introduces Optuna-based hyperparameter optimization. It
defines a tunable classifier whose two hidden layers share a sampled width, then
runs five reproducible trials of 10 epochs each. A seeded TPE sampler searches a
logarithmic learning-rate range from `1e-5` to `1e-1` and a hidden-layer width
from 20 to 300 neurons. Each trial is scored by its best validation accuracy,
after which the study prints the best learning rate, layer width, and score.

## 8. `prob5/script8_optuna_pruning.py` — Prune weak tuning trials

The final script extends the Optuna search to 20 trials and adds a median pruner.
It implements the training and validation loop at the epoch level so that every
trial can report intermediate validation accuracy to Optuna. Trials that are not
promising can be stopped early, avoiding unnecessary training, while completed
trials return their best validation accuracy. The script concludes by printing
the best hyperparameter configuration and validation score found by the pruned
study.

## Overall workflow

Together, the scripts create an incremental pipeline:

1. Download and split Fashion-MNIST.
2. Batch the data for training and evaluation.
3. Construct a baseline neural-network classifier.
4. Train it while recording loss and accuracy.
5. Interpret individual validation predictions.
6. Plot training and validation accuracy.
7. Optimize learning rate and network width.
8. Make that optimization more efficient with early pruning.

The design deliberately reuses earlier modules in later steps, so dataset
preparation, model definitions, and training logic stay consistent throughout
the project.

## Corrections and final packaging

The final review resulted in the following corrections:

- All eight scripts, the dialog summary, and the notebook were consolidated in
  the requested `prob5/` directory.
- Training and validation now use micro-averaged multiclass accuracy, so the
  reported value is the overall fraction of correctly classified images rather
  than a macro average across classes.
- The shuffled training `DataLoader` has its own generator seeded with `42`,
  making its initial batch order reproducible as well as the dataset split and
  model initialization.
- The self-contained notebook,
  `prob5/e89_Hussein_Jawad_HW03_Prob5.ipynb`, presents all eight stages in order
  and trains the baseline model only once. Its prediction and plotting stages
  reuse that trained model and its recorded history instead of repeating the
  same training run.
- This updated dialog summary is included as the notebook's final Markdown cell
  so the submitted notebook records both the completed workflow and the final
  corrections.
